In [ ]:
import re
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.1)
CATEGORY_ORDER = ["entertainment", "fitness", "food", "gaming", "research_science", "tech"]
CAT_PALETTE = dict(zip(CATEGORY_ORDER, sns.color_palette("tab10", 6)))

df = pd.read_csv("../../data/processed/combined_videos_raw.csv", low_memory=False)

# ── Type casting ────────────────────────────────────────────────────────────
df["video_publishedAt"] = pd.to_datetime(df["video_publishedAt"], utc=True, errors="coerce")
df["snapshot_utc"]      = pd.to_datetime(df["snapshot_utc"],      utc=True, errors="coerce")
df["publish_year"]      = df["video_publishedAt"].dt.year
df["publish_month"]     = df["video_publishedAt"].dt.to_period("M")

def parse_duration(val):
    if pd.isna(val):
        return np.nan
    m = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(val).strip())
    if not m:
        return np.nan
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return float(h * 3600 + mi * 60 + s)

df["duration_sec"] = df["duration"].apply(parse_duration)
df["is_short"]     = df["duration_sec"] <= 60

for col in ["views", "likes", "comments"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Shape          : {df.shape}")
print(f"Unique videos  : {df['video_id'].nunique():,}")
print(f"Unique channels: {df['channel_title'].nunique():,}")
print(f"Categories     : {sorted(df['category_name'].unique())}")
print(f"Publish range  : {df['video_publishedAt'].min().date()} → {df['video_publishedAt'].max().date()}")

In [ ]:
# ── Row counts & null rates for key columns ─────────────────────────────────
key_cols = [
    "video_title", "video_description", "video_tags",
    "views", "likes", "comments", "duration_sec",
    "channel_country", "channel_subscriberCount",
]

null_df = pd.DataFrame({
    "null_count": df[key_cols].isna().sum(),
    "null_pct":   (df[key_cols].isna().mean() * 100).round(1),
})
print("=== Null rates for key columns ===")
print(null_df.to_string())

print("\n=== Videos per category ===")
print(df["category_name"].value_counts().reindex(CATEGORY_ORDER).to_string())

print("\n=== Channels per category ===")
print(df.groupby("category_name")["channel_title"].nunique().reindex(CATEGORY_ORDER).to_string())

print("\n=== Videos flagged as Shorts (≤ 60s) ===")
print(df.groupby("category_name")["is_short"].sum().reindex(CATEGORY_ORDER).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All categories together — log-scale histogram
ax = axes[0]
data = df["views"].dropna()
data = data[data > 0]
ax.hist(np.log10(data), bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("log₁₀(views)")
ax.set_ylabel("Video count")
ax.set_title("View count distribution (all categories)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"10^{x:.0f}"))

# Median views by category — horizontal bar
ax2 = axes[1]
medians = (
    df[df["views"] > 0]
    .groupby("category_name")["views"]
    .median()
    .reindex(CATEGORY_ORDER)
    .sort_values()
)
colors = [CAT_PALETTE[c] for c in medians.index]
bars = ax2.barh(medians.index, medians.values, color=colors)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}K"))
ax2.set_xlabel("Median views")
ax2.set_title("Median view count by category")
for bar, val in zip(bars, medians.values):
    ax2.text(val * 1.02, bar.get_y() + bar.get_height() / 2,
            f"{val/1e3:.0f}K", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("../../outputs/figures/views_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Like rate and comment rate (per view), excluding 0-view videos
df["like_rate"]    = df["likes"]    / df["views"].replace(0, np.nan)
df["comment_rate"] = df["comments"] / df["views"].replace(0, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, label in zip(
    axes,
    ["like_rate", "comment_rate"],
    ["Median like rate (likes / view)", "Median comment rate (comments / view)"],
):
    vals = (
        df.groupby("category_name")[metric]
        .median()
        .reindex(CATEGORY_ORDER)
        .sort_values()
    )
    colors = [CAT_PALETTE[c] for c in vals.index]
    bars = ax.barh(vals.index, vals.values * 100, color=colors)
    ax.set_xlabel("Rate (%)")
    ax.set_title(label)
    for bar, val in zip(bars, vals.values):
        ax.text(val * 100 * 1.02, bar.get_y() + bar.get_height() / 2,
                f"{val*100:.2f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("../../outputs/figures/engagement_rates.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution excluding Shorts (≤ 60s) and extreme outliers (> 2h)
long_form = df[(~df["is_short"]) & (df["duration_sec"] <= 7200)]["duration_sec"] / 60

ax = axes[0]
ax.hist(long_form, bins=80, color="darkorange", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Video count")
ax.set_title("Long-form video duration (excl. Shorts, excl. >2h)")
ax.axvline(long_form.median(), color="black", linestyle="--", linewidth=1.2,
           label=f"Median: {long_form.median():.0f} min")
ax.legend()

# Median duration per category (long-form only)
ax2 = axes[1]
med_dur = (
    df[~df["is_short"]]
    .groupby("category_name")["duration_sec"]
    .median()
    .reindex(CATEGORY_ORDER)
    .sort_values() / 60
)
colors = [CAT_PALETTE[c] for c in med_dur.index]
bars = ax2.barh(med_dur.index, med_dur.values, color=colors)
ax2.set_xlabel("Median duration (minutes)")
ax2.set_title("Median long-form duration by category")
for bar, val in zip(bars, med_dur.values):
    ax2.text(val * 1.02, bar.get_y() + bar.get_height() / 2,
             f"{val:.0f} min", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("../../outputs/figures/duration_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nShorts (≤60s) by category:")
print(df.groupby("category_name")["is_short"].agg(["sum","mean"])
        .rename(columns={"sum":"count","mean":"pct"})
        .assign(pct=lambda x: (x["pct"]*100).round(1))
        .reindex(CATEGORY_ORDER).to_string())

In [ ]:
channel_df = df.groupby(["channel_title", "category_name"]).agg(
    videos          = ("video_id",              "count"),
    subscribers     = ("channel_subscriberCount","first"),
    total_views     = ("views",                 "sum"),
    median_views    = ("views",                 "median"),
    median_likes    = ("likes",                 "median"),
    median_comments = ("comments",              "median"),
    shorts_count    = ("is_short",              "sum"),
).reset_index()

print("=== Top 10 channels by median views ===")
print(
    channel_df.nlargest(10, "median_views")
    [["channel_title","category_name","subscribers","median_views","videos"]]
    .to_string(index=False)
)

print("\n=== Subscriber count distribution by category ===")
print(
    channel_df.groupby("category_name")["subscribers"]
    .describe()[["min","25%","50%","75%","max"]]
    .reindex(CATEGORY_ORDER)
    .applymap(lambda x: f"{x:,.0f}")
    .to_string()
)

# Scatter: subscribers vs median views, colored by category
fig, ax = plt.subplots(figsize=(10, 6))
for cat in CATEGORY_ORDER:
    sub = channel_df[channel_df["category_name"] == cat]
    ax.scatter(
        sub["subscribers"], sub["median_views"],
        label=cat, alpha=0.65, s=40, color=CAT_PALETTE[cat]
    )
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Subscribers (log)"); ax.set_ylabel("Median views per video (log)")
ax.set_title("Subscribers vs. Median views per video")
ax.legend(title="Category", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../../outputs/figures/subscribers_vs_views.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Videos published per quarter, 2020 onward
recent = df[df["publish_year"] >= 2020].copy()
recent["quarter"] = recent["video_publishedAt"].dt.to_period("Q")

vol = (
    recent.groupby(["quarter", "category_name"])
    .size()
    .reset_index(name="count")
)
vol["quarter_dt"] = vol["quarter"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 5))
for cat in CATEGORY_ORDER:
    sub = vol[vol["category_name"] == cat].sort_values("quarter_dt")
    ax.plot(sub["quarter_dt"], sub["count"], marker="o", markersize=3,
            label=cat, color=CAT_PALETTE[cat])

ax.set_xlabel("Quarter")
ax.set_ylabel("Videos published")
ax.set_title("Publishing volume per quarter by category (2020–2026)")
ax.legend(title="Category", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../../outputs/figures/publish_volume_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# How much usable text do we have for BERTopic?
df["has_description"] = df["video_description"].notna()
df["has_tags"]        = df["video_tags"].notna()
df["has_both"]        = df["has_description"] & df["has_tags"]

# Title word count
df["title_wordcount"] = df["video_title"].str.split().str.len()

# Description word count (non-null only)
df["desc_wordcount"] = df["video_description"].str.split().str.len()

print("=== Text field coverage by category ===")
coverage = df.groupby("category_name").agg(
    has_description = ("has_description", "mean"),
    has_tags        = ("has_tags",        "mean"),
    has_both        = ("has_both",        "mean"),
    median_title_words = ("title_wordcount", "median"),
    median_desc_words  = ("desc_wordcount",  "median"),
).reindex(CATEGORY_ORDER)
coverage[["has_description","has_tags","has_both"]] = (
    coverage[["has_description","has_tags","has_both"]] * 100
).round(1)
print(coverage.to_string())

# Title word count distribution
fig, ax = plt.subplots(figsize=(10, 4))
for cat in CATEGORY_ORDER:
    sub = df[df["category_name"] == cat]["title_wordcount"].dropna()
    ax.hist(sub, bins=30, alpha=0.5, label=cat, color=CAT_PALETTE[cat], density=True)
ax.set_xlabel("Word count")
ax.set_ylabel("Density")
ax.set_title("Video title word count distribution by category")
ax.legend(title="Category", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../../outputs/figures/title_wordcount.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nVideos with NO description AND NO tags: {(~df['has_description'] & ~df['has_tags']).sum():,}")
print(f"  → will rely on title alone for BERTopic")

In [ ]:
# Collect all quality flags in one place — useful reference before preprocessing
flags = {
    "views == 0":                    (df["views"] == 0).sum(),
    "views null":                    df["views"].isna().sum(),
    "likes null (disabled)":         df["likes"].isna().sum(),
    "comments null (disabled)":      df["comments"].isna().sum(),
    "duration unparseable":          df["duration_sec"].isna().sum(),
    "is_short (≤60s)":               df["is_short"].sum(),
    "duration > 2h":                 (df["duration_sec"] > 7200).sum(),
    "no description":                (~df["has_description"]).sum(),
    "no tags":                       (~df["has_tags"]).sum(),
    "no description AND no tags":    (~df["has_description"] & ~df["has_tags"]).sum(),
    "channel_country missing":       df["channel_country"].isna().sum(),
    "SD definition (not HD)":        (df["definition"] == "sd").sum(),
}

flag_df = pd.DataFrame.from_dict(flags, orient="index", columns=["count"])
flag_df["pct_of_total"] = (flag_df["count"] / len(df) * 100).round(2)
print(f"Total videos: {len(df):,}\n")
print(flag_df.to_string())